# Convolution check (PyTorch)
This notebook shows a 4x4 matrix `A` padded to 6x6, convolved with a 3x3 kernel `K`, producing a 4x4 output. It demonstrates how to get a mathematical convolution (kernel flipped) and verifies results with a manual loop.

In [20]:
import torch
import torch.nn.functional as F
# Large-scale convolution setup (double precision)
C = 3
H = 1024
W = 1024
P = 1  # padding
K_out = 64
FH = FW = 3
# Create input tensor I of shape (1, C, H+2P, W+2P) with double precision
I0 = torch.zeros((1, C, H + 2 * P, W + 2 * P), dtype=torch.float64)
# Build the unpadded inner tensor I[c, x, y] = c * (x + y) for x in [0..W-1], y in [0..H-1]
xs = torch.arange(W, dtype=torch.float64)
ys = torch.arange(H, dtype=torch.float64)
grid = ys.unsqueeze(1) + xs.unsqueeze(0)  # shape (H, W), grid[y,x] = x + y
for c in range(C):
    I0[0, c, P:P+H, P:P+W] = (c * grid)
print('Input I0 shape:', I0.shape, 'dtype:', I0.dtype)
# Define filters with shape (K_out, C, FH, FW) in double precision using F[k,c,i,j] = (c + k) * (i + j)
filters = torch.zeros((K_out, C, FH, FW), dtype=torch.float64)
for k in range(K_out):
    for c in range(C):
        for i in range(FH):
            for j in range(FW):
                filters[k, c, i, j] = float((c + k) * (i + j))
print('Filters shape:', filters.shape, 'dtype:', filters.dtype)

Input I0 shape: torch.Size([1, 3, 1026, 1026]) dtype: torch.float64
Filters shape: torch.Size([64, 3, 3, 3]) dtype: torch.float64


In [21]:
# PyTorch's conv2d performs cross-correlation. To perform a mathematical convolution, rotate each filter spatially by 180 degrees.
F_conv = torch.rot90(filters, 2, dims=(2,3))  # rotate 180 degrees on FHxFW dims for each (k,c)
# Perform convolution (no extra padding, stride=1) -> output shape will be (1, K_out, H, W)
# Note: I0 has shape (1, C, H+2P, W+2P) where P=1, and filters are 3x3 so output spatial dims are H x W
out = F.conv2d(I0, F_conv, padding=0)
print('Output shape (batch, K_out, H_out, W_out):', out.shape)
print('Output dtype:', out.dtype)
# Add a small sampled verification: compare conv2d result against manual sum at a few positions for k=0 and k=1
sample_positions = [(0,0), (10,10), (512,512), (1023,1023)]
check_k = [0, 1]
out_cpu = out.cpu()  # ensure on CPU for indexing if using CUDA
F_conv_cpu = F_conv.cpu()
Icpu = I0.cpu()
for k in check_k:
    for (y, x) in sample_positions:
        # manual convolution at (y,x) for filter k: sum_c sum_i,j I0[c, y+i, x+j] * F_conv[k,c,i,j]
        manual = 0.0
        for c in range(C):
            patch = Icpu[0, c, y:y+FH, x:x+FW]
            manual += (patch * F_conv_cpu[k, c]).sum().item()
        auto_val = out_cpu[0, k, y, x].item()
        print(f'k={k} pos=({y},{x}) manual={manual} conv2d={auto_val} diff={manual - auto_val}')
# Sum all elements of the output tensor and print the result
total_sum = out.sum().item()
print('Total sum of all elements in output tensor:', total_sum)

Output shape (batch, K_out, H_out, W_out): torch.Size([1, 64, 1024, 1024])
Output dtype: torch.float64
k=0 pos=(0,0) manual=10.0 conv2d=10.0 diff=0.0
k=0 pos=(10,10) manual=1740.0 conv2d=1740.0 diff=0.0
k=0 pos=(512,512) manual=92100.0 conv2d=92100.0 diff=0.0
k=0 pos=(1023,1023) manual=122690.0 conv2d=122690.0 diff=0.0
k=1 pos=(0,0) manual=16.0 conv2d=16.0 diff=0.0
k=1 pos=(10,10) manual=2784.0 conv2d=2784.0 diff=0.0
k=1 pos=(512,512) manual=147360.0 conv2d=147360.0 diff=0.0
k=1 pos=(1023,1023) manual=196304.0 conv2d=196304.0 diff=0.0
Total sum of all elements in output tensor: 122756344698240.0


In [19]:
out

tensor([[[[1.0000e+01, 4.0000e+01, 8.5000e+01,  ..., 4.5940e+04,
           4.5985e+04, 4.0910e+04],
          [4.0000e+01, 1.2000e+02, 2.1000e+02,  ..., 9.1920e+04,
           9.2010e+04, 7.6735e+04],
          [8.5000e+01, 2.1000e+02, 3.0000e+02,  ..., 9.2010e+04,
           9.2100e+04, 7.6810e+04],
          ...,
          [4.5940e+04, 9.1920e+04, 9.2010e+04,  ..., 1.8372e+05,
           1.8381e+05, 1.5324e+05],
          [4.5985e+04, 9.2010e+04, 9.2100e+04,  ..., 1.8381e+05,
           1.8390e+05, 1.5331e+05],
          [4.0910e+04, 7.6735e+04, 7.6810e+04,  ..., 1.5324e+05,
           1.5331e+05, 1.2269e+05]],

         [[1.6000e+01, 6.4000e+01, 1.3600e+02,  ..., 7.3504e+04,
           7.3576e+04, 6.5456e+04],
          [6.4000e+01, 1.9200e+02, 3.3600e+02,  ..., 1.4707e+05,
           1.4722e+05, 1.2278e+05],
          [1.3600e+02, 3.3600e+02, 4.8000e+02,  ..., 1.4722e+05,
           1.4736e+05, 1.2290e+05],
          ...,
          [7.3504e+04, 1.4707e+05, 1.4722e+05,  ..., 2.9395

In [ ]:
# Manual verification using nested loops (explicit convolution across channels and multiple filters)
A6s = A6.squeeze()  # shape (3,6,6)
out_manual = torch.zeros((K_out, 4, 4), dtype=torch.float32)
for k in range(K_out):
    for i in range(4):
        for j in range(4):
            total = 0.0
            for c in range(C_in):
                patch = A6s[c, i:i+3, j:j+3]
                total += (patch * F_conv[k, c]).sum()
            out_manual[k, i, j] = total
print('Manual convolution result (K_out x H_out x W_out):')
print(out_manual)
# Compare
print('\nMax absolute difference:', (out_manual - out.squeeze()).abs().max().item())